1. **Creating a Custom Dataset for your files**

    A custom Dataset class must implement three functions: __init__, __len__, and __getitem__. Take a look at this implementation; the SatelliteImageClassification images are stored in a directory img_dir, and their labels are stored separately in a CSV file annotations_file.

    ```
    __init__
    ```
    The \__init__ function is run once when instantiating the Dataset object. We initialize the annotations file, and both transforms 


    ```
    __len__
    ```
    The \__len__ function returns the number of samples in our dataset.


    ```
    __getitem__
    ```
    The \__getitem__ function loads and returns a sample from the dataset at the given index idx. Based on the index, it identifies the image’s location on disk, converts that to a tensor using read_image, retrieves the corresponding label from the csv data in self.img_labels, calls the transform functions on them (if applicable), and returns the tensor image and corresponding label in a tuple.

In [1]:
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2 as transforms

import numpy as np

import os
import cv2
import torch


class VideoDataset(Dataset):
    def __init__(self, root_dir, labels, transform=None, max_frames=0, resize=(224, 224)):
        self.root_dir = root_dir
        self.labels = labels
        self.transform = transform
        self.max_frames = max_frames
        self.resize = resize
        self.video_paths = []
        self.video_labels = []

        for label in labels:
            label_dir = os.path.join(root_dir, label)
            for filename in os.listdir(label_dir):
                if filename.endswith(('.mp4', '.avi', '.mov')):
                    self.video_paths.append(os.path.join(label_dir, filename))
                    self.video_labels.append(label)

    def __len__(self):
        return len(self.video_paths)

    def load_video(self, path):
        cap = cv2.VideoCapture(path)
        frames = []
        try:
            while True:
                ret, frame = cap.read()
                if not ret:
                    break
                if self.resize:
                    frame = cv2.resize(frame, self.resize)
                frames.append(frame)
                if self.max_frames and len(frames) >= self.max_frames:
                    break
        finally:
            cap.release()
        return np.array(frames)

    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = self.video_labels[idx]
        video_frames = self.load_video(video_path)

        if self.transform:
            video_frames = [self.transform(frame) for frame in video_frames]

        # Stack frames to create a tensor
        video_tensor = torch.stack(video_frames)

        # Assuming labels are provided as strings
        label_idx = self.labels.index(label)

        return video_tensor, label_idx

    - Preparing your data for training with DataLoaders

In [2]:

# Parameters
root_dir = './action_classification_dataset/train'
classes = ['Biking', 'Diving', 'HorseRiding']
transform = transforms.Compose([
    transforms.ToImage(),
    transforms.ToDtype(torch.float32),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))

])

batch_size = 4
max_frames = 30
resize = (112, 112)

# Create dataset
train_dataset = VideoDataset(root_dir=root_dir, labels=classes, transform=transform, max_frames=max_frames, resize=resize)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)


    - Show sample video data from the DataLoader

In [3]:
# Example usage
for batch_idx, (videos, labels) in enumerate(train_dataloader):
    print(f'Videos shape: {videos.shape}')
    print(f'Labels shape: {labels.shape}')
    break

Videos shape: torch.Size([4, 30, 3, 112, 112])
Labels shape: torch.Size([4])


2. Define a Convolutional Neural Network

In [4]:
from network.convlstm import ConvLSTM

import torch.nn as nn

class ConvLSTMModel(nn.Module):
    def __init__(self, num_classes, image_size, batch_size=1, hidden_dim=[64, 64]):
        super(ConvLSTMModel, self).__init__()
        """
        input_dim: Number of channels in input
        hidden_dim: Number of hidden channels
        kernel_size: Size of kernel in convolutions
        num_layers: Number of LSTM layers stacked on each other
        batch_first (default=False): Whether or not dimension 0 is the batch or not
        bias (default=True): Bias or no bias in Convolution
        return_all_layers(default=False): Return the list of computations for all layers
        Note: Will do same padding.

        """
        self.batch_size = batch_size
        self.convlstm = ConvLSTM(input_dim=3, hidden_dim=hidden_dim, kernel_size=(3, 3), num_layers=len(hidden_dim), batch_first=True)
        self.fc = nn.Linear(image_size[0] * image_size[1] * hidden_dim[-1], num_classes)

    def forward(self, x):
        # dada shape (batch_size, time, c, h, w)
        _, last_state_list = self.convlstm(x)
    
        # reshape information from last hidden state
        x = last_state_list[-1][0].view(self.batch_size, -1)
        x = self.fc(x)
        return x

# Parameters
num_classes = len(classes)
model = ConvLSTMModel(num_classes, image_size=(112, 112), batch_size=batch_size)

3. Define a Loss function and optimizer

In [5]:
import torch.optim as optim

# construct an optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

4. Train the network

In [6]:
torch.manual_seed(0)

# Check if GPU (cuda) is available
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Move our model to device (GPU or CPU)
model.to(device)
model.train()

num_epochs = 5

for epoch in range(num_epochs):
    running_loss = 0.0
    
    for batch_idx, (videos, labels) in enumerate(train_dataloader):
        # get the inputs; data is a list of [inputs, labels]
        videos, labels = videos.to(device), labels.to(device)

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = model(videos)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        if batch_idx % 30 == 29:  # print every 50 batches
            print(f'Epoch [{epoch + 1}/{num_epochs}], Batch [{batch_idx + 1}/{len(train_dataloader)}], Loss: {running_loss / 30:.4f}')
            running_loss = 0.0

print('Finished Training')

Epoch [1/5], Batch [30/60], Loss: 1.7022
Epoch [1/5], Batch [60/60], Loss: 2.4728
Epoch [2/5], Batch [30/60], Loss: 1.1206
Epoch [2/5], Batch [60/60], Loss: 1.1982
Epoch [3/5], Batch [30/60], Loss: 0.8004
Epoch [3/5], Batch [60/60], Loss: 1.1656
Epoch [4/5], Batch [30/60], Loss: 2.0533
Epoch [4/5], Batch [60/60], Loss: 1.3228
Epoch [5/5], Batch [30/60], Loss: 1.1489
Epoch [5/5], Batch [60/60], Loss: 1.0248
Finished Training


In [7]:
# Save model
torch.save(model.state_dict(), 'video_classification_model.pt')